# 02 - Feature Engineering

Create derived features for ML modeling from the project data.

## Features to Create
- Performance indices (SPI, CPI)
- Variance metrics (budget, schedule)
- Team metrics (stability, productivity)
- Binary risk indicators
- Temporal features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from src.data import FeatureEngineer

print("Modules loaded!")

## 1. Load Data

In [ ]:
# Load raw data
df = pd.read_csv('../data/raw/sample_projects.csv')
print(f"Original shape: {df.shape}")
print(f"Original columns: {len(df.columns)}")
df.head()

## 2. Apply Feature Engineering

In [ ]:
# Create feature engineer and apply
fe = FeatureEngineer()
df_features = fe.create_features(df)

print(f"New shape: {df_features.shape}")
print(f"\nNew features created: {len(fe.get_feature_names())}")
print(f"\nFeature names:")
for name in fe.get_feature_names():
    print(f"  - {name}")

In [ ]:
# View engineered features
engineered_cols = fe.get_feature_names()
df_features[engineered_cols].describe()

## 3. Performance Indices Analysis

In [ ]:
# Schedule Performance Index (SPI) and Cost Performance Index (CPI)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SPI by risk level
if 'schedule_performance_index' in df_features.columns:
    for risk in ['Low', 'Medium', 'High']:
        mask = df_features['risk_level'] == risk
        values = df_features.loc[mask, 'schedule_performance_index']
        axes[0].hist(values, alpha=0.5, label=risk, bins=10)
    axes[0].axvline(x=1.0, color='black', linestyle='--', label='On Schedule')
    axes[0].set_xlabel('SPI (Schedule Performance Index)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('SPI Distribution by Risk Level\n(< 1 = Behind Schedule)')
    axes[0].legend()

# CPI by risk level
if 'cost_performance_index' in df_features.columns:
    for risk in ['Low', 'Medium', 'High']:
        mask = df_features['risk_level'] == risk
        values = df_features.loc[mask, 'cost_performance_index']
        axes[1].hist(values, alpha=0.5, label=risk, bins=10)
    axes[1].axvline(x=1.0, color='black', linestyle='--', label='On Budget')
    axes[1].set_xlabel('CPI (Cost Performance Index)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('CPI Distribution by Risk Level\n(< 1 = Over Budget)')
    axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Performance indices summary by risk
perf_cols = ['schedule_performance_index', 'cost_performance_index', 'budget_variance_pct']
perf_cols = [c for c in perf_cols if c in df_features.columns]

print("Performance Indices by Risk Level:")
print("=" * 50)
print(df_features.groupby('risk_level')[perf_cols].mean().round(3))

## 4. Team Metrics Analysis

In [ ]:
# Team stability vs risk
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Team stability
if 'team_stability' in df_features.columns:
    sns.boxplot(data=df_features, x='risk_level', y='team_stability', 
               order=['Low', 'Medium', 'High'],
               palette={'Low': 'green', 'Medium': 'orange', 'High': 'red'},
               ax=axes[0])
    axes[0].set_title('Team Stability by Risk Level\n(Higher = More Stable)')

# Team productivity (if available)
if 'velocity' in df_features.columns:
    sns.boxplot(data=df_features, x='risk_level', y='velocity', 
               order=['Low', 'Medium', 'High'],
               palette={'Low': 'green', 'Medium': 'orange', 'High': 'red'},
               ax=axes[1])
    axes[1].set_title('Velocity by Risk Level')

plt.tight_layout()
plt.show()

## 5. Feature Correlations

In [ ]:
# Correlation of engineered features with risk
df_features['risk_numeric'] = df_features['risk_level'].map({'Low': 0, 'Medium': 1, 'High': 2})

# Get available engineered columns
eng_cols = [c for c in engineered_cols if c in df_features.columns and df_features[c].notna().sum() > 0]

correlations = df_features[eng_cols + ['risk_numeric']].corr()['risk_numeric'].drop('risk_numeric')
correlations = correlations.sort_values(ascending=False)

# Plot
plt.figure(figsize=(10, 6))
colors = ['red' if x > 0 else 'green' for x in correlations]
correlations.plot(kind='barh', color=colors)
plt.xlabel('Correlation with Risk Level')
plt.title('Feature Correlations with Risk\n(Red = Higher value → Higher risk)')
plt.axvline(x=0, color='black', linestyle='-')
plt.tight_layout()
plt.show()

print("\nCorrelation values:")
print(correlations.round(3))

In [ ]:
# Feature correlation heatmap
if len(eng_cols) > 1:
    plt.figure(figsize=(12, 10))
    corr_matrix = df_features[eng_cols].corr()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
               cmap='RdYlGn_r', center=0, vmin=-1, vmax=1)
    plt.title('Engineered Features Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 6. Binary Risk Indicators

In [ ]:
# Check binary indicators if present
binary_cols = ['is_over_budget', 'is_behind_schedule', 'is_high_complexity']
binary_cols = [c for c in binary_cols if c in df_features.columns]

if binary_cols:
    print("Binary Indicator Breakdown:")
    print("=" * 50)
    for col in binary_cols:
        counts = df_features[col].value_counts()
        print(f"\n{col}:")
        print(f"  True: {counts.get(1, 0)} projects")
        print(f"  False: {counts.get(0, 0)} projects")
        
        # Risk distribution for True cases
        if counts.get(1, 0) > 0:
            true_risks = df_features[df_features[col] == 1]['risk_level'].value_counts()
            print(f"  Risk breakdown when True: {true_risks.to_dict()}")

## 7. Feature Selection Recommendations

In [ ]:
# Select features based on correlation and availability
print("FEATURE SELECTION RECOMMENDATIONS")
print("=" * 60)

# High correlation features (absolute value > 0.3)
high_corr = correlations[abs(correlations) > 0.3]
print(f"\n🎯 High Correlation Features (|r| > 0.3):")
for feat, corr in high_corr.items():
    direction = '↑' if corr > 0 else '↓'
    print(f"   {direction} {feat}: {corr:.3f}")

# Medium correlation features
med_corr = correlations[(abs(correlations) > 0.1) & (abs(correlations) <= 0.3)]
print(f"\n📊 Medium Correlation Features (0.1 < |r| <= 0.3):")
for feat, corr in med_corr.items():
    direction = '↑' if corr > 0 else '↓'
    print(f"   {direction} {feat}: {corr:.3f}")

# Recommended feature set for ML
print(f"\n✅ Recommended Feature Set for ML:")
recommended = correlations[abs(correlations) > 0.1].index.tolist()
for feat in recommended:
    print(f"   - {feat}")

## 8. Save Engineered Data

In [ ]:
# Save engineered features
output_path = '../data/processed/projects_with_features.csv'
df_features.to_csv(output_path, index=False)
print(f"✅ Saved engineered data to: {output_path}")
print(f"   Shape: {df_features.shape}")

## Summary

### Features Created:
1. **Performance Indices**: SPI, CPI (measures efficiency)
2. **Variance Metrics**: Budget variance %, schedule variance
3. **Team Metrics**: Team stability (1 - turnover)
4. **Temporal Features**: Days since start, days remaining, project duration

### Key Insights:
- Lower SPI/CPI correlates with higher risk
- Team stability is important for risk prediction
- Budget variance is a strong predictor

### Next Steps:
- Use these features for ML modeling in `03_ml_modeling.ipynb`